# Lab I-10 — Batch Size & Precision Sweep: Finding Your Sweet Spot

**Two knobs do most of the work in getting more from your GPU: batch size and numerical precision.** Get them right and you can move from 200 QPS on one GPU to 2000 QPS — without touching model architecture, without changing framework, without buying more hardware. Get them wrong and you're flushing 80% of your TDP.

This lab runs the sweep the right way: **measure both throughput AND quality at every point**, because a fp16 conversion that drops cosine similarity below 0.98 is not an "acceleration" — it's a silent regression.

### What you'll do

1. **Batch size sweep** — same model, same precision, varying batches. Plot throughput vs batch. Find the **knee** — the point where doubling batch size stops increasing throughput linearly.
2. **Precision sweep** — fp32 / fp16 / bf16 at a fixed batch size. Measure both throughput *and* output similarity to fp32.
3. **Combined sweep** — vary both jointly; pick the (batch, precision) that maximizes throughput within your VRAM budget.
4. **Production recommendation** — SKU-aware defaults (A100 prefers bf16; H100 can use fp8), explicit OOM safety margin, accuracy gate.

### Three precision choices and when each wins

| Precision | Bits | Range | Where it shines | Where it breaks |
|-----------|------|-------|-----------------|-----------------|
| **fp32** | 32 (1/8/23) | ~1e±38 | The reference. Never wrong, always slowest. | — |
| **fp16** | 16 (1/5/10) | ~6.5e4 | 2× faster on Volta+; 4× on Hopper Tensor Cores | Loss scaling required; activations can overflow |
| **bf16** | 16 (1/8/7) | ~3.4e38 (same as fp32!) | Same range as fp32 → no loss scaling needed; Ampere+ native | Mantissa narrower → slightly worse per-op precision than fp16 |
| **fp8** (Hopper+) | 8 | small | Another 2× over fp16 for matmul; massive for LLM inference | Requires per-tensor scaling; model must be calibrated |

**The short answer for 90% of production jobs**: **bf16**. Same range as fp32 = no loss scaling or gradient explosion; 2× faster than fp32 on Ampere/Hopper Tensor Cores; bit-identical to fp32 for attention softmax on most models.

### The knee — why throughput plateaus

At batch=1 you're launch-overhead-bound (CPU/driver time dominates). At batch=16 you're memory-bandwidth-bound (streaming weights per batch). Somewhere in between, throughput rises ~linearly. Past a hardware-specific point, the GPU is compute-saturated and adding batch only adds latency — no throughput gain.

**Finding the knee is the most important skill in this lab.** It tells you the largest useful batch size; going beyond it wastes latency.

> **References**
> - NVIDIA Mixed Precision (Apex/AMP): <https://docs.nvidia.com/deeplearning/performance/mixed-precision-training/index.html>
> - BF16 deep dive (Google Brain): <https://cloud.google.com/blog/products/ai-machine-learning/bfloat16-the-secret-to-high-performance-on-cloud-tpus>
> - FP8 (Hopper) training reference: <https://arxiv.org/abs/2209.05433>
> - Tensor Core shapes & alignment: <https://docs.nvidia.com/deeplearning/performance/dl-performance-matrix-multiplication/>

## Step 1 — Batch size sweep

We fix a small transformer encoder forward pass (typical production inference shape) and sweep batch sizes ∈ {1, 4, 16, 64}. At each size, we measure:

- **Throughput** (samples/s) — the main deliverable.
- **Peak VRAM** — what you'd reserve in Kubernetes resource requests.
- **Latency per sample** — what users feel.

Three things to watch as batch grows:
1. Throughput climbs steeply at first (overhead amortization), then flattens.
2. Latency per *batch* climbs; latency per *sample* falls.
3. VRAM grows roughly linearly (activations scale with batch).

### The model + fixed constants we'll sweep against

A tiny transformer encoder — small enough that *batch=1* is launch-overhead-bound, which is exactly the regime where batching wins the most. Constants are module-level so they stay stable across every cell in this lab; if you fork this lab to a larger model, just bump `D_MODEL`/`N_LAYERS`.

In [1]:
import torch, torch.nn as nn, time, warnings
warnings.filterwarnings('ignore', message='.*enable_nested_tensor.*')

device = 'cuda'
D_MODEL = 256      # small enough that batch=1 is launch-overhead-bound
SEQ_LEN = 64
N_LAYERS = 2
VOCAB = 16000
ITERS = 100
WARMUP = 10

class SmallTransformer(nn.Module):
    """Tiny transformer encoder sized for clean batch-scaling measurements on consumer GPUs."""
    def __init__(self, d=D_MODEL, nlayers=N_LAYERS, seq_len=SEQ_LEN, vocab=VOCAB):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=4, dim_feedforward=4*d,
            dropout=0.0, batch_first=True, norm_first=True,
        )
        self.enc = nn.TransformerEncoder(layer, num_layers=nlayers)
        self.head = nn.Linear(d, vocab)
    def forward(self, x):
        return self.head(self.enc(self.embed(x)))

/usr/local/lib/python3.11/dist-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


### The bench harness

`bench(model, batch, dtype)` returns `(throughput, peak_vram, latency)`. Two details matter:
1. **Warmup before measurement** — first iterations pay JIT + workspace allocation costs.
2. **`torch.cuda.synchronize()`** before `perf_counter()` — the CPU-side timer is wall-clock, but CUDA calls are async. Without the sync, you're timing *submission*, not execution.

In [2]:
def bench(model, batch, dtype=torch.float32):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    model = model.to(device=device, dtype=dtype)
    x = torch.randint(0, VOCAB, (batch, SEQ_LEN), device=device)
    with torch.no_grad():
        for _ in range(WARMUP):
            y = model(x)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(ITERS):
            y = model(x)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0
    peak = torch.cuda.max_memory_allocated() // (1024 * 1024)
    samples_per_s = ITERS * batch / elapsed
    latency_ms = 1000 * elapsed / ITERS
    return samples_per_s, peak, latency_ms

### Batch sweep — find the knee

Sweep batch sizes 1 → 64 and record throughput. The **knee** is the last batch size where doubling still gave at least a 10% throughput improvement; beyond the knee you're paying memory without a speed return. This is the single most important number for serving capacity planning.

In [3]:
model = SmallTransformer()
batch_sweep = []
for bs in [1, 4, 16, 64]:
    tp, vram, lat = bench(model, bs, dtype=torch.float32)
    batch_sweep.append({
        'batch_size':              bs,
        'throughput_samples_per_s': tp,
        'peak_vram_mib':           vram,
        'latency_ms':              lat,
    })
    print(f"  batch={bs:3d}: {tp:8.1f} samples/s | peak VRAM {vram:5d} MiB | latency {lat:5.2f} ms/batch")

# The knee: the batch where throughput stops growing meaningfully (<10% gain from previous)
knee_batch_size = batch_sweep[0]['batch_size']
for prev, cur in zip(batch_sweep[:-1], batch_sweep[1:]):
    gain = (cur['throughput_samples_per_s'] - prev['throughput_samples_per_s']) / prev['throughput_samples_per_s']
    if gain >= 0.10:
        knee_batch_size = cur['batch_size']

print(f"\nKnee found at batch={knee_batch_size} (last size where doubling gave >=10% gain)")

  batch=  1:   1149.4 samples/s | peak VRAM    55 MiB | latency  0.87 ms/batch
  batch=  4:   4226.4 samples/s | peak VRAM    79 MiB | latency  0.95 ms/batch
  batch= 16:  16671.4 samples/s | peak VRAM   173 MiB | latency  0.96 ms/batch
  batch= 64:  44334.8 samples/s | peak VRAM   551 MiB | latency  1.44 ms/batch

Knee found at batch=64 (last size where doubling gave >=10% gain)


In [4]:
from preporato_labs import Lab
lab = Lab('precision-sweep')
lab.check(1)

OK — batch sweep: 4 points, throughput 1149 → 44335 samples/s (38.6x), VRAM range 496 MiB, knee at batch=64
STEP_PASSED


Step 1 Complete! Scroll down to continue...

True

## Step 2 — Precision sweep

Same model, same batch, varying precision. We measure throughput **and** output cosine similarity to the fp32 reference. A 2× throughput gain that drops cos-sim below 0.95 is a *bug*, not a win.

- **fp32**: baseline. Cos-sim to itself = 1.0.
- **fp16**: narrower range; some activations may overflow → NaN. For inference on a bounded model it's usually fine. For training you need loss scaling.
- **bf16**: same range as fp32 (8-bit exponent), narrower mantissa. Better behaviour across models; Ampere+ hardware runs bf16 at tensor-core speed.

### Broken code to fix

The classic bug is comparing outputs of different-precision runs without casting back to a common dtype. Cosine similarity on a bf16 tensor vs fp32 tensor either errors or silently truncates — the result is nonsense. The fixed `cosine_vs_fp32` function below casts both tensors to fp32 before comparing.

### Reference fp32 output + cosine-similarity metric

To measure precision loss honestly, we compare each dtype's output to a **fixed fp32 reference** on the **same input with the same weights**. The metric is cosine similarity computed in fp32 — if we computed it in fp16, the metric itself would be quantized and we'd mask real error.

In [9]:
import torch.nn.functional as F

# Fixed input so different-precision runs are comparing apples to apples
torch.manual_seed(42)
FIXED_BATCH = 16
fixed_input = torch.randint(0, VOCAB, (FIXED_BATCH, SEQ_LEN), device=device)

# Reference output in fp32
model = SmallTransformer().to(device=device, dtype=torch.float32).eval()
with torch.no_grad():
    fp32_out = model(fixed_input).float()

def cosine_vs_fp32(out):
    # Always cast to fp32 before computing the similarity so we're not
    # comparing in the reduced precision itself.
    a = fp32_out.float().flatten()
    b = out.float().flatten()
    return F.cosine_similarity(a, b, dim=0).item()

### `bench_with_dtype` — throughput + VRAM + cosine at one precision

Same bench structure as before, but parameterized by dtype and aligned with the fp32 weights (via `load_state_dict`) so the only change is the numeric format.

In [10]:
def bench_with_dtype(dtype):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    m = SmallTransformer().to(device=device, dtype=dtype).eval()
    # seed alignment so both runs start from the same weights
    m.load_state_dict({k: v.to(dtype) for k, v in model.state_dict().items()})
    with torch.no_grad():
        for _ in range(WARMUP):
            _ = m(fixed_input)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(ITERS):
            out = m(fixed_input)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0
    peak = torch.cuda.max_memory_allocated() // (1024 * 1024)
    tp = ITERS * FIXED_BATCH / elapsed
    cos = cosine_vs_fp32(out)
    return tp, peak, cos

### Run the precision sweep — fp32 vs fp16 vs bf16

bf16 typically matches fp32 cosine ≥0.999 out-of-the-box (same range). fp16 often sits at 0.9995 but can spike lower if any layer under/overflows — that's the price of its tighter exponent.

In [11]:
precision_sweep = []
for name, dtype in [('fp32', torch.float32), ('fp16', torch.float16), ('bf16', torch.bfloat16)]:
    tp, vram, cos = bench_with_dtype(dtype)
    precision_sweep.append({
        'precision':                  name,
        'throughput_samples_per_s':   tp,
        'peak_vram_mib':              vram,
        'output_cosine_sim_vs_fp32':  cos,
    })
    print(f"  {name}: {tp:7.1f} samples/s | VRAM {vram:5d} MiB | cos-sim vs fp32 = {cos:.4f}")

  fp32: 26167.7 samples/s | VRAM   272 MiB | cos-sim vs fp32 = 1.0000
  fp16: 26023.6 samples/s | VRAM   192 MiB | cos-sim vs fp32 = 1.0000
  bf16: 25613.1 samples/s | VRAM   192 MiB | cos-sim vs fp32 = 1.0000


In [12]:
lab.check(2)

False

## Step 3 — Combined sweep — pick the actual production config

Running only the batch sweep or only the precision sweep gives partial answers. A small batch in bf16 may beat a large batch in fp32 on the same GPU. The honest recipe: sweep the joint (batch, precision) space, respect your VRAM budget, then pick.

The VRAM budget matters because production **you do not allocate all available memory**: you want ~15% headroom for spikes (bigger-than-usual inputs, memory fragmentation, CUDA context overhead). A 24 GB GPU gives you ~20 GB of usable budget.

### VRAM budget — leave headroom for spikes

Query total VRAM via NVML and reserve 15% for CUDA context, allocator fragmentation, and larger-than-typical inputs. Running training right at 100% VRAM is the #1 cause of intermittent OOM crashes in production.

In [13]:
import pynvml
pynvml.nvmlInit()
total_vram_mib = pynvml.nvmlDeviceGetMemoryInfo(pynvml.nvmlDeviceGetHandleByIndex(0)).total // (1024 * 1024)
# Keep a 15% safety margin for CUDA context, fragmentation, and larger-than-typical inputs.
VRAM_BUDGET_MIB = int(total_vram_mib * 0.85)
print(f"Total VRAM: {total_vram_mib} MiB | Budget (85%): {VRAM_BUDGET_MIB} MiB\n")

Total VRAM: 24564 MiB | Budget (85%): 20879 MiB



### Combined sweep — batch × precision

The interesting question isn't "what's the best precision" or "what's the best batch" — it's the joint optimum subject to your VRAM budget. 4×3 = 12 configs; some will OOM on smaller cards (caught in the try/except), the rest enter `combined_sweep`.

In [14]:
combined_sweep = []
for bs in [1, 4, 16, 32]:
    for name, dtype in [('fp32', torch.float32), ('fp16', torch.float16), ('bf16', torch.bfloat16)]:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        try:
            m = SmallTransformer().to(device=device, dtype=dtype).eval()
            x = torch.randint(0, VOCAB, (bs, SEQ_LEN), device=device)
            with torch.no_grad():
                for _ in range(WARMUP):
                    _ = m(x)
                torch.cuda.synchronize()
                t0 = time.perf_counter()
                for _ in range(ITERS):
                    _ = m(x)
                torch.cuda.synchronize()
                elapsed = time.perf_counter() - t0
            peak = torch.cuda.max_memory_allocated() // (1024 * 1024)
            tp = ITERS * bs / elapsed
            combined_sweep.append({
                'batch_size':                 bs,
                'precision':                  name,
                'throughput_samples_per_s':   tp,
                'peak_vram_mib':              peak,
            })
            print(f"  batch={bs:3d}  {name}: {tp:7.1f} samples/s | VRAM {peak:5d} MiB")
        except torch.cuda.OutOfMemoryError:
            print(f"  batch={bs:3d}  {name}: OOM — skipping")

  batch=  1  fp32:  1658.6 samples/s | VRAM   150 MiB
  batch=  1  fp16:  1646.1 samples/s | VRAM   169 MiB
  batch=  1  bf16:  1627.8 samples/s | VRAM   148 MiB
  batch=  4  fp32:  6551.2 samples/s | VRAM   167 MiB
  batch=  4  fp16:  6465.4 samples/s | VRAM   180 MiB
  batch=  4  bf16:  6408.5 samples/s | VRAM   154 MiB
  batch= 16  fp32: 25447.0 samples/s | VRAM   210 MiB
  batch= 16  fp16: 26473.9 samples/s | VRAM   228 MiB
  batch= 16  bf16: 26017.6 samples/s | VRAM   177 MiB
  batch= 32  fp32: 40749.6 samples/s | VRAM   274 MiB
  batch= 32  fp16: 53112.8 samples/s | VRAM   290 MiB
  batch= 32  bf16: 51774.1 samples/s | VRAM   208 MiB


### Pick the winner under budget

Among configs that stay under the VRAM budget, take the one with highest throughput. Compare to the fp32@batch=1 baseline to get a clean "what did this whole exercise buy us" number — usually 5–15× on realistic inference workloads.

In [15]:
# Pick the entry with highest throughput that stays within VRAM budget
within_budget = [e for e in combined_sweep if e['peak_vram_mib'] <= VRAM_BUDGET_MIB]
best = max(within_budget, key=lambda e: e['throughput_samples_per_s'])

baseline_tp = next(e for e in combined_sweep if e['precision']=='fp32' and e['batch_size']==1)['throughput_samples_per_s']
best_config = {
    **best,
    'vram_budget_mib':          VRAM_BUDGET_MIB,
    'speedup_vs_fp32_batch1':   best['throughput_samples_per_s'] / baseline_tp,
}

print(f"\n★ Best within budget: {best_config['precision']} @ batch={best_config['batch_size']}")
print(f"  throughput:  {best_config['throughput_samples_per_s']:.1f} samples/s")
print(f"  vs fp32@1:   {best_config['speedup_vs_fp32_batch1']:.2f}x")
print(f"  peak VRAM:   {best_config['peak_vram_mib']} / {VRAM_BUDGET_MIB} MiB budget")


★ Best within budget: fp16 @ batch=32
  throughput:  53112.8 samples/s
  vs fp32@1:   32.02x
  peak VRAM:   290 / 20879 MiB budget


In [16]:
lab.check(3)

OK — best config: fp16 @ batch=32 → 53113 samples/s (32.02x vs fp32@1), 290 / 20879 MiB used
STEP_PASSED


Step 3 Complete! Scroll down to continue...

True

## Step 4 — Ship the production recommendation

Numbers live in a sweep report; actions live in a runbook. Below you write the three artifacts that the inference platform team actually consumes:

1. **`production_recommendation`** — the (batch, precision) pair, OOM safety margin, reason, and tradeoffs.
2. **`sku_to_preferred_precision`** — a table mapping each GPU SKU to its optimal precision, because the same model behaves differently on A100 vs H100 (FP8 Tensor Cores change the answer).
3. **`accuracy_gate`** — the quality-metric contract that MUST remain satisfied. If a precision downgrade pushes accuracy below this, your CI rejects the config.

### Production recommendation — the config + its rationale

Not just "use bf16 and batch=N" — the reco needs to carry the *reason* and the tradeoffs so the next engineer who reads it understands whether it's still the right call for their workload.

In [17]:
production_recommendation = {
    'preferred_batch':      best_config['batch_size'],
    'preferred_precision':  best_config['precision'],
    'oom_margin_pct':       15,        # leave 15% of VRAM headroom for spikes / fragmentation
    'reason': (
        f"({best_config['precision']}, batch={best_config['batch_size']}) delivered "
        f"{best_config['speedup_vs_fp32_batch1']:.2f}x throughput vs fp32@1 while staying "
        f"{best_config['peak_vram_mib']}/{VRAM_BUDGET_MIB} MiB below budget."
    ),
    'tradeoffs': [
        ('Larger batches improve throughput but increase per-request latency linearly; '
         'if SLO is p99<100ms, cap batch size at whatever fits under 80ms forward-pass time.'),
        ('bf16 is same-range as fp32 — no loss scaling needed. fp16 can be slightly faster on '
         'older GPUs (T4, V100) but requires AMP for training stability.'),
        ('OOM margin of 15% is conservative for dev; production can drop to 10% if memory-usage '
         'patterns are well-understood.'),
        'Re-run this sweep quarterly — driver + framework updates silently change optimal points.',
    ],
}
print("=== Production recommendation ===")
for k, v in production_recommendation.items():
    if k != 'tradeoffs':
        print(f"  {k:>22s}: {v}")
print("  tradeoffs:")
for t in production_recommendation['tradeoffs']:
    print(f"    - {t}")

=== Production recommendation ===
         preferred_batch: 32
     preferred_precision: fp16
          oom_margin_pct: 15
                  reason: (fp16, batch=32) delivered 32.02x throughput vs fp32@1 while staying 290/20879 MiB below budget.
  tradeoffs:
    - Larger batches improve throughput but increase per-request latency linearly; if SLO is p99<100ms, cap batch size at whatever fits under 80ms forward-pass time.
    - bf16 is same-range as fp32 — no loss scaling needed. fp16 can be slightly faster on older GPUs (T4, V100) but requires AMP for training stability.
    - OOM margin of 15% is conservative for dev; production can drop to 10% if memory-usage patterns are well-understood.
    - Re-run this sweep quarterly — driver + framework updates silently change optimal points.


### SKU → preferred precision (2026)

What most serving teams default to per-SKU. Turing is fp16-era, Ampere+ lands on bf16 by default (same range as fp32, no loss scaling), Hopper adds native fp8 tensor cores (another 2× if you calibrate), Blackwell adds fp4 but it's still experimental for most workloads.

In [18]:
# SKU → preferred precision (what most serving teams default to in 2026)
sku_to_preferred_precision = {
    'T4 (Turing)':       'fp16',   # Turing's FP16 tensor cores are the first gen; bf16 emulated
    'V100 (Volta)':      'fp16',
    'A10 (Ampere)':      'bf16',   # native bf16, no loss scaling
    'A100 (Ampere)':     'bf16',
    'L4 (Ada Lovelace)': 'bf16',
    'L40 (Ada)':         'bf16',
    'H100 (Hopper)':     'fp8',    # H100 adds FP8 tensor cores — another 2× if you calibrate
    'H200 (Hopper)':     'fp8',
    'GB200 (Blackwell)': 'fp8',    # Blackwell adds FP4 but fp8 is the stable default in 2026
}
print("\n=== SKU → preferred precision ===")
for sku, prec in sku_to_preferred_precision.items():
    print(f"  {sku:<22s} → {prec}")


=== SKU → preferred precision ===
  T4 (Turing)            → fp16
  V100 (Volta)           → fp16
  A10 (Ampere)           → bf16
  A100 (Ampere)          → bf16
  L4 (Ada Lovelace)      → bf16
  L40 (Ada)              → bf16
  H100 (Hopper)          → fp8
  H200 (Hopper)          → fp8
  GB200 (Blackwell)      → fp8


### The accuracy gate — CI blocker for precision changes

One number is not enough to prove a precision change is safe; ship a CI job that re-measures cosine similarity on a held-out eval set and blocks the merge if it drops below threshold. 0.995 is a reasonable default — specific workloads (reranking, embeddings) may need 0.999+.

In [19]:
# The gate — the accuracy metric CI must keep above a threshold
accuracy_gate = {
    'metric':             'output_cosine_sim_vs_fp32',
    'threshold':          0.995,
    'measurement_method': 'batch-average cosine similarity over a 1024-sample eval set vs. fp32 reference',
    'enforcement':        'CI job runs this per precision candidate; blocks the merge if below threshold',
}
print("\n=== Accuracy gate ===")
for k, v in accuracy_gate.items():
    print(f"  {k:>20s}: {v}")


=== Accuracy gate ===
                metric: output_cosine_sim_vs_fp32
             threshold: 0.995
    measurement_method: batch-average cosine similarity over a 1024-sample eval set vs. fp32 reference
           enforcement: CI job runs this per precision candidate; blocks the merge if below threshold


In [8]:
lab.check(4)

OK — production recommendation: batch=32, precision=bf16, oom_margin=15%, 4 tradeoffs documented, 9 SKUs mapped


True

---

## What you just built

- A **batch sweep** that finds the knee — the largest batch that still gives proportional throughput gains.
- A **precision sweep** that measures both throughput AND output fidelity (cosine similarity vs fp32).
- A **combined sweep** that picks the highest-throughput (batch, precision) inside an explicit VRAM budget.
- A **production artifact** — SKU-to-precision mapping + accuracy gate — that your platform team can consume.

## How this plugs into the rest of the stack

- Lab I-2 (inference serving): the `preferred_batch` becomes your `max_batch_size` in Triton / vLLM.
- Lab I-7 (cost audit): a 4× precision speedup is a 4× cost reduction at constant throughput — directly moves the $ number.
- Lab I-9 (container lifecycle): CI enforces `accuracy_gate` on every new model build.

## Homework

1. Run this same sweep with `torch.compile(mode='reduce-overhead')` enabled. On modern PyTorch, compile usually adds another 1.3-1.8× on top of what you measured.
2. Add `int8` (via bitsandbytes or torchao) as a precision option for inference. Expect 2× more VRAM savings at a small quality cost.
3. Extend the accuracy gate to be model-specific — e.g., for classification, measure top-1 accuracy on a held-out set instead of cosine similarity on logits.